In [4]:
"""
Etosha National Park — Food Web Lotka-Volterra Simulation
==========================================================
Reads:  immc_-_species_final__3_.csv
        immc_-_foodweb_final__5_.csv
 
Model design
------------
All species fall into one of four ODE regimes:
 
1. PLANTS (pure producer)
   dN/dt = r_plant * N * (1 - N/K) * season(t)
           - sum_i F[i, PLANTS] * N[i]
 
2. BUGS (colony producer — logistic, no metabolic death)
   dN/dt = r_bug * N * (1 - N/K) * season(t)
           - sum_i F[i, BUGS] * N[i]
   (Insects are modelled like a renewable resource, not an individual animal.)
 
3. PANGOLIN & AARDVARK (specialist insectivores — slow logistic + predation)
   dN/dt = r_intr * N * (1 - N/K) * season(t)
           - sum_k F[k, i] * N[k]
   These are decoupled from Bug dynamics to avoid the boom-bust instability that
   arises when insectivore populations scale proportionally to insect abundance.
 
4. All other consumers (herbivores, omnivores, carnivores)
   dN/dt = sum_j E[i,j] * F[i,j] * N[i]    ← energy gained from prey
           - sum_k F[k,i] * N[k]            ← lost to predators
           - x[i] * N[i]                    ← allometric metabolic death
           - delta[i] * N[i]^2              ← weak intra-specific competition
                                               (allows oscillations without
                                                infinite population growth)
 
Functional response (Type II for all predator-prey links):
   F[i,j] = A[i,j] * N[j] / (1 + A[i,j] * H[i,j] * N[j])
 
Attack rate calibration
-----------------------
For each consumer-resource pair, A is set so that the per-capita energy gain
equals TARGET_RATIO × metabolic cost at the initial population:
 
   E[i,j] * F[i,j] = TARGET_RATIO * x[i] / E[i,j]   →   solve for A[i,j]
 
TARGET_RATIO = 1.3 gives a slight surplus → populations grow slowly →
predator populations lag behind → classic LV oscillations emerge.
 
For carnivores, a single a0_carn scalar is solved numerically (brentq) so that
LION's gain/meta = TARGET_RATIO, then applied allometrically to all carnivores.
 
Seasonal forcing
----------------
season(t) = 1 + 0.40 * sin(2π t / 12 − π/2)
Etosha wet season peaks ~February, dry season ~August.
 
Parameters
----------
r_plant   = 3.5   yr⁻¹   grass net primary productivity
r_bug     = 10.0  yr⁻¹   insect colony turnover
r_pangol  = 0.15  yr⁻¹   pangolin intrinsic growth
r_aardva  = 0.12  yr⁻¹   aardvark intrinsic growth
x0        = 0.04         metabolic scalar  (x[i] = x0 * M[i]^-0.25)
h0        = 0.5          handling-time scalar
delta[i]  = x[i] / (8 * K[i])   weak quadratic mortality
"""
 
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from collections import defaultdict
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
 
# ─────────────────────────────────────────────────────────────────────────────
# 1.  LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
 
species_df  = pd.read_csv('species.csv')
foodweb_df  = pd.read_csv('foodweb.csv')
 
CODES   = list(species_df['code'])
NAMES   = list(species_df['species'])
TYPES   = list(species_df['type'])
MASSES  = np.array(species_df['body_mass_kg'], dtype=float)
 
# Initial populations (fill missing producers with reasonable defaults)
N0_raw  = species_df['initial_pop'].fillna(0).to_numpy(dtype=float).copy()
N0_raw[CODES.index('PLANTS')] = 800_000.0    # 40% of K so seasonal swings are visible
N0_raw[CODES.index('BUGS')]   = 2_000_000.0
 
n   = len(CODES)
idx = {c: i for i, c in enumerate(CODES)}
 
# Build prey list per predator from the food web CSV
pp = defaultdict(list)
for _, row in foodweb_df.iterrows():
    pp[row['pred_code']].append(row['prey_code'])
 
# ─────────────────────────────────────────────────────────────────────────────
# 2.  ECOLOGICAL PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
 
r_plant  = 3.5     # grass growth rate  (yr⁻¹)
r_bug    = 10.0    # insect colony turnover (yr⁻¹)
r_pangol = 0.15    # pangolin intrinsic growth
r_aardva = 0.12    # aardvark intrinsic growth
 
x0  = 0.04         # metabolic loss scalar
h0  = 0.5          # handling-time scalar
 
# Allometric metabolic death rate: x[i] = x0 * M^(-1/4)
x_rate = x0 * MASSES**(-0.25)
 
# Realistic carrying capacities (animals: ~1.5–2× census counts)
K_dict = {
    'PLANTS': 2_000_000, 'BUGS': 5_000_000,
    'AEPMEL': 8_000,  'ALCBUS': 1_000,  'CONTAU': 10_000,
    'EQUBUR': 25_000, 'GIRCAM': 5_000,  'LOXAFR': 5_000,
    'MADKIR': 8_000,  'PHAACT': 8_000,  'PROCAP': 30_000,
    'TAUORY': 5_000,  'ORYXOR': 8_000,  'BRHINO': 1_500,
    'WRHINO': 2_500,  'ANTELO': 25_000,
    'PANGOL': 15_000, 'AARDVA': 25_000,
    'PANLEO': 800,    'PANPAR': 700,
    'CROCRO': 800,    'ACIJUB': 400,
    'CARCAR': 300,    'LEPSER': 600,
}
K_arr = np.array([K_dict.get(c, 5_000) for c in CODES], dtype=float)
 
# Weak intra-specific competition: δ[i] = x[i] / (8 × K[i])
# At N = K this adds only x[i]/8 extra mortality — gentle enough to allow
# oscillations but prevents runaway growth.
delta = x_rate / (8.0 * K_arr)
 
# ─────────────────────────────────────────────────────────────────────────────
# 3.  BUILD ATTACK-RATE (A), HANDLING-TIME (H), EFFICIENCY (E) MATRICES
# ─────────────────────────────────────────────────────────────────────────────
 
TARGET_RATIO = 1.3   # desired gain/metabolic-cost ratio at initial populations
                     # >1 so populations grow → predator lag → oscillations
 
A_mat = np.zeros((n, n))
H_mat = np.zeros((n, n))
E_mat = np.zeros((n, n))
 
# — Handling times (allometric, capped to avoid numerical issues) —
for pred, prey_list in pp.items():
    i = idx[pred]
    for prey in prey_list:
        j = idx[prey]
        H_mat[i, j] = min(h0 * MASSES[j]**0.75 * MASSES[i]**(-0.5), 500.0)
 
# — A for plant/bug consumers: per-species metabolic calibration —
#   Set A[i,j] so that E*F*N[i] = TARGET_RATIO * x[i] * N[i]  at t=0
for pred, prey_list in pp.items():
    i = idx[pred]
    if TYPES[i] not in ('herbivore', 'insectivore', 'omnivore'):
        continue
    # Count how many plant/bug prey items this species has
    n_base_prey = sum(
        1 for q in prey_list
        if TYPES[idx[q]] == 'producer' or q == 'BUGS'
    )
    if n_base_prey == 0:
        continue
    for prey in prey_list:
        j = idx[prey]
        if TYPES[j] != 'producer' and prey != 'BUGS':
            continue
        pref  = 1.0 / len(prey_list)   # equal diet preference
        e_val = 0.80                    # plant assimilation efficiency
        # Desired per-capita functional response from this prey item
        F_des = (TARGET_RATIO * x_rate[i] / e_val) / n_base_prey
        Np    = N0_raw[j]
        # Solve F = A*Np/(1+A*H*Np) = F_des  →  A = F_des/(Np - F_des*H*Np)
        denom = Np - F_des * H_mat[i, j] * Np
        A_ij  = F_des / denom if denom > 1e-9 else F_des / Np
        A_mat[i, j] = max(A_ij, 1e-20) * pref
        E_mat[i, j] = e_val
 
# — a0_carn: solve so LION's gain/meta = TARGET_RATIO, apply allometrically —
i_lion = idx['PANLEO']
n_lion_prey = len(pp['PANLEO'])
 
def lion_gain_minus_target(a0):
    gain = 0.0
    for prey in pp['PANLEO']:
        j    = idx[prey]
        pref = 1.0 / n_lion_prey
        A_ij = a0 * (MASSES[i_lion] / max(MASSES[j], 0.001))**0.25 * pref
        F    = A_ij * N0_raw[j] / max(1 + A_ij * H_mat[i_lion, j] * N0_raw[j], 1e-15)
        gain += 0.45 * F * N0_raw[i_lion]
    return gain / (x_rate[i_lion] * N0_raw[i_lion]) - TARGET_RATIO
 
a0_carn = brentq(lion_gain_minus_target, 1e-10, 5.0)
 
# Apply allometric attack rate to all predator→prey edges not already set
for pred, prey_list in pp.items():
    i = idx[pred]
    for prey in prey_list:
        j = idx[prey]
        if A_mat[i, j] > 0:
            continue  # already calibrated (herbivore eating plants)
        pref = 1.0 / len(prey_list)
        A_mat[i, j] = a0_carn * (MASSES[i] / max(MASSES[j], 0.001))**0.25 * pref
        E_mat[i, j] = 0.45 if TYPES[i] == 'carnivore' else 0.60
 
# ─────────────────────────────────────────────────────────────────────────────
# 4.  ODE SYSTEM
# ─────────────────────────────────────────────────────────────────────────────
 
# Special-case indices
IP = idx['PLANTS']
IB = idx['BUGS']
IPG = idx['PANGOL']
IAA = idx['AARDVA']
 
# Set of producer indices (PLANTS + BUGS — both treated as renewable resources)
PRODUCER_IDX = {IP, IB}
# Set of logistic-only indices (pangolin, aardvark — slow breeders, no boom-bust)
LOGISTIC_IDX = {IPG, IAA}
 
 
def season_full(t):
    """Full ±40% seasonal forcing for plants and herbivores.
    Etosha wet peak ~Feb, dry trough ~Aug."""
    return 1.0 + 0.40 * np.sin(2.0 * np.pi * t / 12.0 - np.pi / 2.0)
 
def season_mild(t):
    """Mild ±10% forcing for insectivores.
    Termite mounds are thermally buffered underground and don't swing
    as hard as above-ground grass, so pangolin/aardvark reproduction
    is much less season-sensitive."""
    return 1.0 + 0.10 * np.sin(2.0 * np.pi * t / 12.0 - np.pi / 2.0)
 
 
def food_web_ode(t, N):
    N  = np.maximum(N, 0.0)
    dN = np.zeros(n)
    sf = season_full(t)
    sm = season_mild(t)
 
    # — Type-II functional responses —
    F = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if A_mat[i, j] > 0:
                F[i, j] = (A_mat[i, j] * N[j]
                           / max(1.0 + A_mat[i, j] * H_mat[i, j] * N[j], 1e-15))
 
    # — Producers: logistic growth × full season − herbivory —
    for i in PRODUCER_IDX:
        r_i = r_plant if i == IP else r_bug
        K_i = K_arr[i]
        dN[i] = (r_i * N[i] * (1.0 - N[i] / K_i) * sf
                 - sum(F[k, i] * N[k] for k in range(n) if A_mat[k, i] > 0))
 
    # — Pangolin & Aardvark: slow logistic × mild season − predation only —
    for i, r_i in [(IPG, r_pangol), (IAA, r_aardva)]:
        dN[i] = (r_i * N[i] * (1.0 - N[i] / K_arr[i]) * sm
                 - sum(F[k, i] * N[k] for k in range(n) if A_mat[k, i] > 0))
 
    # — All other consumers: LV gain − predation − metabolic − weak density —
    for i in range(n):
        if i in PRODUCER_IDX or i in LOGISTIC_IDX:
            continue
        gain   = sum(E_mat[i, j] * F[i, j] * N[i]
                     for j in range(n) if A_mat[i, j] > 0)
        loss_p = sum(F[k, i] * N[k]
                     for k in range(n) if A_mat[k, i] > 0)
        dN[i]  = gain - loss_p - x_rate[i] * N[i] - delta[i] * N[i]**2
 
    return dN
 
# ─────────────────────────────────────────────────────────────────────────────
# 5.  SOLVE
# ─────────────────────────────────────────────────────────────────────────────
 
T_YEARS  = 80
T_MONTHS = T_YEARS * 12
t_eval   = np.linspace(0, T_MONTHS, T_MONTHS * 20)   # 20 points per month
 
print("Solving Lotka-Volterra system …")
print(f"  Species:  {n}")
print(f"  Duration: {T_YEARS} years  ({T_MONTHS} months)")
print(f"  t_eval points: {len(t_eval):,}")
 
sol = solve_ivp(
    food_web_ode,
    (0, T_MONTHS),
    N0_raw.copy(),
    method='RK45',
    t_eval=t_eval,
    rtol=1e-6,
    atol=1e-8,
    max_step=0.2,        # ≤ 0.2 months for accuracy with seasonal forcing
)
 
print(f"  Solver status: {'OK ✓' if sol.success else sol.message}")
 
t_yr = sol.t / 12.0
Y    = np.maximum(sol.y, 0.0)
 
# ─────────────────────────────────────────────────────────────────────────────
# 6.  SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
 
print(f"\n{'Species':32s}  {'Year 0':>10s}  {'Year 40':>10s}  {'Year 80':>10s}  Status")
print("─" * 78)
for i in range(n):
    v0  = Y[i,  0]
    v40 = Y[i, len(t_eval)//2]
    v80 = Y[i, -1]
    status = "EXTINCT" if v80 < 1 else "stable"
    print(f"  {NAMES[i]:30s}  {v0:>10,.0f}  {v40:>10,.0f}  {v80:>10,.0f}  {status}")
 
survivors = sum(1 for i in range(n) if Y[i, -1] >= 1)
print(f"\nSurvivors at year 80: {survivors}/{n}")
 
# ─────────────────────────────────────────────────────────────────────────────
# 7.  PLOT
# ─────────────────────────────────────────────────────────────────────────────
 
BG   = '#0d1117'
CELL = '#161b22'
GRID = '#21262d'
 
# Species groups with display colours
GROUPS = {
    'Producers': {
        'codes':  ['PLANTS', 'BUGS'],
        'colors': ['#2e7d32', '#66bb6a'],
    },
    'Megaherbivores': {
        'codes':  ['LOXAFR', 'GIRCAM', 'BRHINO', 'WRHINO'],
        'colors': ['#1565c0', '#42a5f5', '#0288d1', '#4dd0e1'],
    },
    'Large Herbivores': {
        'codes':  ['EQUBUR', 'CONTAU', 'TAUORY', 'ANTELO', 'ORYXOR', 'ALCBUS'],
        'colors': ['#6a1b9a', '#ab47bc', '#ce93d8', '#e040fb', '#ba68c8', '#7b1fa2'],
    },
    'Small Herbivores': {
        'codes':  ['AEPMEL', 'MADKIR', 'PROCAP', 'PHAACT'],
        'colors': ['#e65100', '#ff7043', '#ffb74d', '#bcaaa4'],
    },
    'Insectivores': {
        'codes':  ['PANGOL', 'AARDVA'],
        'colors': ['#bf360c', '#ff8a65'],
    },
    'Carnivores': {
        'codes':  ['PANLEO', 'PANPAR', 'CROCRO', 'ACIJUB', 'CARCAR', 'LEPSER'],
        'colors': ['#b71c1c', '#e53935', '#ef9a9a', '#ff8f00', '#ffd600', '#4dd0e1'],
    },
}
 
fig = plt.figure(figsize=(22, 26), facecolor=BG)
fig.suptitle(
    'Etosha National Park — Food Web Population Dynamics (80 Years)\n'
    'Lotka–Volterra · Allometric Calibration · Seasonal Forcing · Weak Density Dependence',
    fontsize=16, color='white', y=0.985, fontweight='bold'
)
 
gs = gridspec.GridSpec(
    3, 2, figure=fig,
    hspace=0.44, wspace=0.30,
    left=0.07, right=0.97, top=0.94, bottom=0.04
)
 
for ax_idx, (gname, gdata) in enumerate(GROUPS.items()):
    ax = fig.add_subplot(gs[ax_idx // 2, ax_idx % 2])
    ax.set_facecolor(CELL)
    for sp in ax.spines.values():
        sp.set_edgecolor('#30363d')
 
    for k, code in enumerate(gdata['codes']):
        i   = idx[code]
        col = gdata['colors'][k % len(gdata['colors'])]
        ax.plot(t_yr, Y[i], color=col, linewidth=1.8, label=NAMES[i], alpha=0.93)
 
    # Shade last 20 years to highlight steady-state / periodic regime
    ax.axvspan(60, 80, color='white', alpha=0.04)
    ax.axvline(60, color='white', alpha=0.15, linewidth=0.8, linestyle='--')
 
    ax.set_title(gname, color='#e6edf3', fontsize=12, fontweight='bold', pad=7)
    ax.set_xlabel('Year', color='#8b949e', fontsize=9)
    ax.set_ylabel('Population', color='#8b949e', fontsize=9)
    ax.tick_params(colors='#8b949e', labelsize=8)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _:
            f'{v/1e6:.1f}M' if v >= 1e6 else
            f'{v/1e3:.0f}k' if v >= 1_000 else
            f'{v:.0f}')
    )
    ax.legend(
        fontsize=7.5, framealpha=0.25, labelcolor='#e6edf3',
        facecolor='#21262d', edgecolor='#30363d',
        loc='upper right', ncol=1
    )
    ax.grid(True, color=GRID, linewidth=0.6, alpha=0.8)
    ax.set_xlim(0, T_YEARS)
 
out_path = 'etosha_lv_80yr.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print(f"\nPlot saved → {out_path}")



Solving Lotka-Volterra system …
  Species:  24
  Duration: 80 years  (960 months)
  t_eval points: 19,200
  Solver status: OK ✓

Species                               Year 0     Year 40     Year 80  Status
──────────────────────────────────────────────────────────────────────────────
  Plants                             800,000   1,993,059   1,992,831  stable
  Ants/Termites                    2,000,000   4,999,599   4,999,597  stable
  Cheetah                                150       1,738         851  stable
  Black-Faced Impala                   2,500       4,431       3,687  stable
  Hartebeest                             160       1,008       1,034  stable
  Caracal                                 50       5,339       5,338  stable
  Wildebeest                           3,000       5,867       5,300  stable
  Spotted/Brown Hyena                    400      11,647      11,091  stable
  Plains Zebra                        15,000      67,958      73,832  stable
  Giraffe             